In [ ]:
import pandas as pd
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [20]:
seed = 42
torch.random.manual_seed(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
root_path = "/home/stefan/ioai-prep/kits/concat_img_cat_cnt"

batch_size = 64

# Data

In [21]:
class CategoryCountDataset(Dataset):
    MAX_W = 32 * 8

    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["ImagePath"])
        image = Image.open(img_path).convert("RGB")

        original_w = image.width
        image = self._resize_strip(image)

        tiles = [image[:, :, i * 32 : (i + 1) * 32] for i in range(8)]
        tensor = torch.stack(tiles, dim=0)

        if self.transform:
            tensor = self.transform(tensor)

        if self.is_test:
            num_valid_tiles = (original_w + 31) // 32
            return tensor, row["SampleID"], num_valid_tiles

        label = torch.tensor(row["Label"], dtype=torch.long)
        return tensor, label

    def __len__(self):
        return len(self.df)

    def _resize_strip(self, img: Image.Image) -> torch.Tensor:
        w, h = img.size
        if h != 32:
            raise ValueError("Height must be 32 px")
        if w > self.MAX_W:
            img = img.crop((0, 0, self.MAX_W, 32))
        elif w < self.MAX_W:
            new = Image.new("RGB", (self.MAX_W, 32), 0)
            new.paste(img, (0, 0))
            img = new
        return transforms.ToTensor()(img)

In [22]:
df_train = pd.read_csv(os.path.join(root_path, "train.csv"))
df_train, df_val = train_test_split(df_train, test_size=0.2, random_state=42)

transform = transforms.Compose(
    [
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

train_dataset = CategoryCountDataset(df_train, root_path, transform=transform)
val_dataset = CategoryCountDataset(df_val, root_path, transform=transform)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=10
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=10
)

In [23]:
# sanity check
x = next(iter(train_loader))
[y.shape for y in x]

[torch.Size([64, 8, 3, 32, 32]), torch.Size([64])]

# Model

In [24]:
model = torch.hub.load(
    "chenyaofo/pytorch-cifar-models", "cifar10_resnet56", pretrained=True
).to(device)
model.eval()

Using cache found in /home/stefan/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


CifarResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias

# Submission

In [25]:
df_test = pd.read_csv(os.path.join(root_path, "test.csv"))
test_dataset = CategoryCountDataset(
    df_test, root_path, transform=transform, is_test=True
)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [26]:
predictions = []

with torch.no_grad():
    for images, sample_ids, num_valid_tiles_batch in tqdm(test_loader):
        images = images.to(device)

        B, N = images.shape[:2]
        flat = images.view(B * N, 3, 32, 32)

        logits = model(flat)
        preds = logits.argmax(1).cpu()

        for b in range(B):
            num_valid = min(int(num_valid_tiles_batch[b].item()), N)

            if num_valid > 0:
                tile_preds = preds[b * N : b * N + num_valid]
                distinct_classes = len(set(tile_preds.tolist()))
                num_categories = min(distinct_classes, num_valid)
            else:
                num_categories = 1

            predictions.append(
                {
                    "SampleID": int(sample_ids[b]),
                    "PredictedLabel": num_categories,
                }
            )

100%|██████████| 2/2 [00:00<00:00, 19.64it/s]


In [27]:
submission_df = pd.DataFrame(predictions)
submission_df.to_csv(os.path.join(root_path, "submission.csv"), index=False)

submission_df.head()

,SampleID,PredictedLabel
0,362,4
1,74,3
2,375,3
3,156,5
4,105,4
